[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/06_%E5%9D%87%E5%8B%BB%E5%88%86%E4%BD%88.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：公車固定每 10 分鐘一班，「等 2～5 分鐘」的機率是多少？班距變 20 分鐘後答案怎麼變？**

- **用在流程哪裡**：隨機抽樣、參數搜尋範圍的建模，以及「完全公平」的基準情境。
- **讀完要能做到**：能說出均勻模型的假設，用區間長度比計算機率。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[數據結構](../名詞解釋/02_數據結構/README.md)、[特徵工程](../名詞解釋/04_特徵工程/README.md)

# 06｜均勻分布：等長區間有相同機率

均勻分布 $U(a,b)$ 說：在 $[a,b]$ 內，**等長的區間機率相同**。PDF 是平的：$f(x) = \frac{1}{b-a}$（區間內），區間外為 0；均值 $\frac{a+b}{2}$，變異數 $\frac{(b-a)^2}{12}$。

**一、情境先於公式：公車例子何時成立？**

等待時間能寫成 $U(0,10)$ 需要兩個假設：(1) 車準時每 10 分鐘一班；(2) 你到站時刻在班距內完全隨機。**理論密度 0.1 不是「等 3 分鐘的機率是 10%」**——連續單點機率永遠是 0。車常誤點、尖峰離峰不均，模型就不適用，換模型前先講清楚假設。

**二、機率＝長度比（連續版的數格子）**

$$P(2 \le X \le 5) = \frac{5-2}{10-0} = \frac{3}{10} = 30\%$$

三種算法互相驗證：長度比、CDFS 相減（`cdf(5)-cdf(2)`）、大量模擬比例。本篇圖把 2～5 塗成橘色面積，面積 0.3 就是答案。班距拉到 20 分鐘，密度降到 0.05，同一問題機率剩 15%——**分母變了，答案跟著變**，和 01 的基準率是同一個道理。

**三、SciPy 的參數陷阱**：`stats.uniform(loc, scale)` 中 `loc` 是下界 $a$，`scale` 是**寬度** $b-a$（不是上界）。所以 $U(5,15)$ 要寫 `loc=5, scale=10`，直接把 15 填進 scale 是本章最常見的程式錯誤。

均勻抽樣是我們**選定**的抽樣方式（模擬、網格搜尋），不代表真實資料均勻——分布模型要先合情境，再用資料檢查。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜先把公車情境的假設說清楚

**為什麼做**：假設車輛準時每 10 分鐘一班，而且到站時刻在班距內均勻分布，等待時間才可建模為 U(0,10)。

**怎麼看**：理論密度為 0.1，不是每個精確等待時間都有 10% 機率；車輛誤點或到站時刻不均勻時，模型可能不適用。

In [ ]:
waits = np.random.default_rng(42).uniform(0, 10, size=2000)
x = np.linspace(-1, 11, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(waits, bins=20, density=True, color=GREEN, alpha=.3, label='模擬等待時間')
ax.plot(x, stats.uniform.pdf(x, loc=0, scale=10), color=ORANGE, lw=2, label='理論密度')
ax.set(title='樣本有波動，理論密度在區間內平坦', xlabel='等待時間（分鐘）', ylabel='機率密度')
ax.legend()
plt.show()
print(f'樣本平均：{waits.mean():.2f}；理論平均：5.00')

## 02｜機率等於區間長度占比

**為什麼做**：2～5 的寬度是 3，整個模型區間的寬度是 10，因此機率是 3/10。

**怎麼看**：橘色面積是 0.3；不同精確值的單點機率仍是 0。CDF 的結果與長度比、模擬比例可以對照。

In [ ]:
a, b = 2, 5
probability = stats.uniform.cdf(b, 0, 10) - stats.uniform.cdf(a, 0, 10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([0, 0, 10, 10], [0, .1, .1, 0], color=GREEN, lw=2)
ax.fill_between([a, b], [.1, .1], color=ORANGE, alpha=.4)
ax.set(xlim=(-.5, 10.5), ylim=(0, .14), xlabel='等待時間（分鐘）',
       ylabel='機率密度', title=f'等 {a}～{b} 分鐘：區間機率 {probability:.0%}')
plt.show()
print(f'模擬比例：{np.mean((waits >= a) & (waits <= b)):.2%}')
assert np.isclose(probability, .3)

## 03｜班距變了，區間機率也變了

**為什麼做**：比較 10 與 20 分鐘班距，保持想問的等待區間不變。

**怎麼看**：班距變長後，密度降低；完整區間面積仍為 1。scale 是區間寬度，若下界不為 0，不能把上界直接當 scale。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(-1, 21, 500)
for gap, color in [(10, GREEN), (20, PURPLE)]:
    ax.plot(x, stats.uniform.pdf(x, loc=0, scale=gap), color=color, label=f'班距 {gap} 分鐘')
    chance = stats.uniform.cdf(5, 0, gap) - stats.uniform.cdf(2, 0, gap)
    print(f'班距 {gap} 分鐘，等 2～5 分鐘的機率：{chance:.0%}')
ax.set(title='區間愈寬，單位區間的機率愈少', xlabel='等待時間（分鐘）', ylabel='機率密度')
ax.legend()
plt.show()

## 接回機器學習

均勻抽樣可用來產生模擬值或在參數範圍內做搜尋；它是我們選定的抽樣方式，不代表真實資料一定均勻。分布模型須符合情境，再用資料檢查。

## 換你解讀

1. 為什麼說「等長區間」，而不說「每個精確值機率相同且非零」？
2. 車輛常誤點，還能直接套 U(0,10) 嗎？
3. U(5,15) 在 SciPy 的 loc、scale 應填什麼？

### 請 AI 幫你做一個小實驗

請 AI 改成 U(5,15)，並計算等待 7～10 分鐘的機率。先寫出區間寬度，再確認 loc 與 scale。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)